In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.lines import Line2D

#Trayectoria deseada 
R = 1.0
def phi(x, y):
    return x**2 + y**2 - R**2

def grad_phi(x, y):
    return np.array([2*x, 2*y])

#Primer obstaculo: elipse inferior
ox1, oy1 = 0.0, -1.0
a, b = 1.0, 0.5
beta = 0.0

def varphi1(x, y):
    X = (x - ox1)*np.cos(beta) + (y - oy1)*np.sin(beta)
    Y = (x - ox1)*np.sin(beta) - (y - oy1)*np.cos(beta)
    return X**2/a**2 + Y**2/b**2 - 1

def grad_varphi1(x, y):     #gradiente de varphi1
    X = (x - ox1)*np.cos(beta) + (y - oy1)*np.sin(beta)
    Y = (x - ox1)*np.sin(beta) - (y - oy1)*np.cos(beta)

    dX = np.array([np.cos(beta), np.sin(beta)])
    dY = np.array([np.sin(beta), -np.cos(beta)])

    return 2*X/a**2*dX + 2*Y/b**2*dY

#Segundo obstáculo: elipse superior
ox2, oy2 = 0.0, 1.0
def varphi2(x, y):
    X = (x - ox2)*np.cos(beta) + (y - oy2)*np.sin(beta)
    Y = (x - ox2)*np.sin(beta) - (y - oy2)*np.cos(beta)
    return X**2/a**2 + Y**2/b**2 - 1

def grad_varphi2(x, y):     #gradiente de varphi2
    X = (x - ox2)*np.cos(beta) + (y - oy2)*np.sin(beta)
    Y = (x - ox2)*np.sin(beta) - (y - oy2)*np.cos(beta)

    dX = np.array([np.cos(beta), np.sin(beta)])
    dY = np.array([np.sin(beta), -np.cos(beta)])

    return 2*X/a**2*dX + 2*Y/b**2*dY
    
#Tercer obstáculo: circunferencia
ox3, oy3 = 1.0, 0.0
a3,b3= 0.5,0.5

def varphi3(x, y):
    X=(x - ox3)*np.cos(beta) + (y - oy3)*np.sin(beta)
    Y=(x - ox3)*np.sin(beta) - (y - oy3)*np.cos(beta)
    
    return X**2/a3**2 + Y**2/b3**2 - 1

def grad_varphi3(x, y):     #gradiente de varphi3
    X=(x - ox3)*np.cos(beta) + (y - oy3)*np.sin(beta)
    Y=(x - ox3)*np.sin(beta) - (y - oy3)*np.cos(beta)

    dX = np.array([np.cos(beta), np.sin(beta)])
    dY = np.array([np.sin(beta), -np.cos(beta)])

    return 2*X/a3**2*dX + 2*Y/b3**2*dY

#Funciones campana para los distintos obstáculos
c = -0.4
l1 = 0.1
l2 = 0.1

def t_Q1(x, y):
    v = varphi1(x, y)
    if v <= c:
        return 0.0
    return np.exp(l1 / (c - v))

def u_R1(x, y):
    v = varphi1(x, y)
    if v >= 0:
        return 0.0
    return np.exp(l2 / v)

def t_Q2(x, y):
    v = varphi2(x, y)
    if v <= c:
        return 0.0
    return np.exp(l1 / (c - v))

def u_R2(x, y):
    v = varphi2(x, y)
    if v >= 0:
        return 0.0
    return np.exp(l2 / v)
    
def t_Q3(x, y):
    v = varphi3(x, y)
    if v <= c:
        return 0.0
    return np.exp(l1 / (c - v))

def u_R3(x, y):
    v = varphi3(x, y)
    if v >= 0:
        return 0.0
    return np.exp(l2 / v)

#Campos vectoriales asociados al camino y a los obstáculos
kp = 1.0
kr = 1.0
E = np.array([[0, -1],
              [1,  0]])

def normalize(v):
    n = np.linalg.norm(v)
    if n < 1e-8:
        return np.zeros_like(v)
    return v / n

def chi_P(x, y):
    g = grad_phi(x, y)
    return E @ g - kp * phi(x, y) * g

def chi_R1(x, y):
    g = grad_varphi1(x, y)
    return E @ g - kr * varphi1(x, y) * g

def chi_R2(x, y):
    g = grad_varphi2(x, y)
    return E @ g - kr * varphi2(x, y) * g

def chi_R3(x, y):
    g = grad_varphi3(x, y)
    return E @ g - kr * varphi3(x, y) * g

#Campo vectorial compuesto
def chi_composite(t, z):
    x, y = z

    wP = t_Q1(x, y) * t_Q2(x, y)*t_Q3(x, y)
    wR1 = u_R1(x, y)
    wR2 = u_R2(x, y)
    wR3 = u_R3(x, y) 

    return (
        wP * normalize(chi_P(x, y))
        + wR1 * normalize(chi_R1(x, y))
        + wR2 * normalize(chi_R2(x, y))
        + wR3 * normalize(chi_R3(x, y))
    )

#Condiciones iniciales desde las que parte el robot
initial_conditions = [
    [np.sqrt(0.6), -1],
    [np.sqrt(0.6), 1],
    [-0.3, -0.1],
    [1,0.4],
    [-0.3, -1.1],
    [0,1.5],
    [0,0]
]

#Intervalo de integración de la ecuación diferencial.
t_span = (0, 40)
t_eval = np.linspace(t_span[0], t_span[1], 3000)

solutions = []

#Integración de la ecuacion diferencial.
for z0 in initial_conditions:
    sol = solve_ivp(
        chi_composite,
        t_span,
        z0,
        t_eval=t_eval,
        rtol=1e-8,
        atol=1e-10
    )
    solutions.append(sol)
    
# Se crea una malla fina de puntos en el intervalo [-2, 2] para x e y.
x = np.linspace(-2, 2, 400)
y = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(x, y)

# Se evalúan las funciones escalares phi y varphi_i sobre toda la malla.
Phi = phi(X, Y)
Varphi1 = varphi1(X, Y)
Varphi2 = varphi2(X, Y)
Varphi3 = varphi3(X, Y)

# Se crea una segunda malla más gruesa para dibujar el campo vectorial y que no se solapen las flechas entre sí.
xq = np.linspace(-2, 2, 35)
yq = np.linspace(-2, 2, 35)
Xq, Yq = np.meshgrid(xq, yq)

# Se inicializan las componentes del campo vectorial.
U = np.zeros_like(Xq)
V = np.zeros_like(Yq)

#Se evalua ne cada punto de la malla gruesa el campo vectorial compuesto, lo que devuelve la velocidad en cada punto.
for i in range(Xq.shape[0]):
    for j in range(Xq.shape[1]):
        dx, dy = chi_composite(0, [Xq[i, j], Yq[i, j]])
        U[i, j] = dx
        V[i, j] = dy

#Se calcula la norma del vector velocidad en cada punto.
N = np.sqrt(U**2 + V**2)

#Se normalizan las componentes U y V para que todas las flechas tengan longitud parecida.
U_plot = np.divide(U, N, out=np.zeros_like(U), where=N > 1e-8)
V_plot = np.divide(V, N, out=np.zeros_like(V), where=N > 1e-8)

# Se define una longitud fija para las flechas del campo vectorial.
arrow_length = 0.09
U_plot *= arrow_length
V_plot *= arrow_length

# Se dibuja el campo vectorial en cada punto (Xq,Yq).
plt.figure(figsize=(8, 8))
plt.quiver(
    Xq, Yq, U_plot, V_plot,
    angles='xy',
    scale_units='xy',
    scale=1,
    width=0.002,
    headwidth=3.5,
    headlength=4.5,
    headaxislength=4,
    pivot='middle',
    color='blue',
    alpha=0.75
)

#Dibujo del camino deseado.
plt.contour(X, Y, Phi, levels=[0], colors='red', linewidths=2)

#Dibujo de las zonas reactivas de los obstáculos.
plt.contour(X, Y, Varphi1, levels=[0], colors='green', linewidths=2)
plt.contour(X, Y, Varphi2, levels=[0], colors='green', linewidths=2)
plt.contour(X, Y, Varphi3, levels=[0], colors='green', linewidths=2)

#Dibujo de las zonas repulsivas de los obstáculos.
plt.contour(X, Y, Varphi1, levels=[c], colors='black', linestyles='--', linewidths=2)
plt.contour(X, Y, Varphi2, levels=[c], colors='black', linestyles='--', linewidths=2)
plt.contour(X, Y, Varphi3, levels=[c], colors='black', linestyles='--', linewidths=2)

#Se dibujan las trayectorias obtenidas previamente con solve_ivp.
for sol in solutions:
    plt.plot(sol.y[0], sol.y[1], linewidth=2)

#Se dibujan con puntos las condiciones iniciales.
for z0 in initial_conditions:
    plt.plot(z0[0], z0[1], 'ko', markersize=4)

ax = plt.gca()
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)

plt.grid(True)
plt.xlabel('$x (m)$', fontsize=16)
plt.ylabel('$y (m)$', fontsize=16)
plt.title('Trayectorias del robot para distintas condiciones iniciales', fontsize=20)
leyenda = [
    Line2D([0], [0], color="red", lw=2, label="Trayectoria"),
    Line2D([0], [0], color="green", lw=2, label="Zonas reactivas"),
    Line2D([0], [0], color="black", lw=2, linestyle="--", label="Zonas repulsivas"),
]

plt.legend(handles=leyenda,fontsize=12)
plt.savefig("simulacion.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.lines import Line2D

#Trayectoria deseada.
ox=oy=0
a, b = 2.0, 1.0

def phi(x, y):
    X = (x - ox)*np.cos(beta) + (y - oy)*np.sin(beta)
    Y = (x - ox)*np.sin(beta) - (y - oy)*np.cos(beta)
    
    return X**2/a**2 + Y**2/b**2 - 1

def grad_phi(x, y):
    X = (x - ox)*np.cos(beta) + (y - oy)*np.sin(beta)
    Y = (x - ox)*np.sin(beta) - (y - oy)*np.cos(beta)

    dX = np.array([np.cos(beta), np.sin(beta)])
    dY = np.array([np.sin(beta), -np.cos(beta)])

    return 2*X/a**2*dX + 2*Y/b**2*dY

#Primer obstaculo: elipse inferior.
ox1, oy1 = 0.0, -1.0
a1, b1 = 1.0, 0.5
beta = 0.0

def varphi1(x, y):
    X = (x - ox1)*np.cos(beta) + (y - oy1)*np.sin(beta)
    Y = (x - ox1)*np.sin(beta) - (y - oy1)*np.cos(beta)
    return X**2/a1**2 + Y**2/b1**2 - 1

def grad_varphi1(x, y):
    X = (x - ox1)*np.cos(beta) + (y - oy1)*np.sin(beta)
    Y = (x - ox1)*np.sin(beta) - (y - oy1)*np.cos(beta)

    dX = np.array([np.cos(beta), np.sin(beta)])
    dY = np.array([np.sin(beta), -np.cos(beta)])

    return 2*X/a**2*dX + 2*Y/b**2*dY

#Segundo obstaculo: óvalo de Cassini
def varphi2(x, y):
    return ((x-0.9)**2+(y-1)**2)*((x+0.9)**2+(y-1)**2)-0.9

def grad_varphi2(x, y):
    A = (x - 0.9)**2 + (y - 1)**2
    B = (x + 0.9)**2 + (y - 1)**2

    dphidx = 2*(x - 0.9)*B + 2*(x + 0.9)*A
    dphidy = 2*(y - 1)*(A + B)

    return np.array([dphidx, dphidy])

# Tercer obstáculo: circunferencia
ox3, oy3 = 2.0, 0.0
a3,b3= 0.5,0.5

def varphi3(x, y):
    X=(x - ox3)*np.cos(beta) + (y - oy3)*np.sin(beta)
    Y=(x - ox3)*np.sin(beta) - (y - oy3)*np.cos(beta)
    
    return X**2/a3**2 + Y**2/b3**2 - 1

def grad_varphi3(x, y):
    X=(x - ox3)*np.cos(beta) + (y - oy3)*np.sin(beta)
    Y=(x - ox3)*np.sin(beta) - (y - oy3)*np.cos(beta)

    dX = np.array([np.cos(beta), np.sin(beta)])
    dY = np.array([np.sin(beta), -np.cos(beta)])

    return 2*X/a3**2*dX + 2*Y/b3**2*dY

#Funciones campana para los distintos obstáculos.
c = -0.4
c2=-0.2
l1 = 0.05
l2 = 0.05

def t_Q1(x, y):
    v = varphi1(x, y)
    if v <= c:
        return 0.0
    return np.exp(l1 / (c - v))

def u_R1(x, y):
    v = varphi1(x, y)
    if v >= 0:
        return 0.0
    return np.exp(l2 / v)

def t_Q2(x, y):
    v = varphi2(x, y)
    if v <= c2:
        return 0.0
    return np.exp(l1 / (c2 - v))

def u_R2(x, y):
    v = varphi2(x, y)
    if v >= 0:
        return 0.0
    return np.exp(l2 / v)
    
def t_Q3(x, y):
    v = varphi3(x, y)
    if v <= c:
        return 0.0
    return np.exp(l1 / (c - v))

def u_R3(x, y):
    v = varphi3(x, y)
    if v >= 0:
        return 0.0
    return np.exp(l2 / v)


#Campos vectoriales asociados al camino y a los obstáculos.
kp = 1.0
kr = 1.0
kr2=3.0
E = np.array([[0, -1],
              [1,  0]])

def normalize(v):
    n = np.linalg.norm(v)
    if n < 1e-8:
        return np.zeros_like(v)
    return v / n

def chi_P(x, y):
    g = grad_phi(x, y)
    return E @ g - kp * phi(x, y) * g

def chi_R1(x, y):
    g = grad_varphi1(x, y)
    return E @ g - kr * varphi1(x, y) * g

def chi_R2(x, y):
    g = grad_varphi2(x, y)
    return E @ g - kr2 * varphi2(x, y) * g

def chi_R3(x, y):
    g = grad_varphi3(x, y)
    return E @ g - kr * varphi3(x, y) * g

#Campo vectorial compuesto.
def chi_composite(t, z):
    x, y = z

    wP = t_Q1(x, y) * t_Q2(x, y)*t_Q3(x, y)
    wR1 = u_R1(x, y)
    wR2 = u_R2(x, y)
    wR3 = u_R3(x, y) # Añado el peso para el tercer obstáculo.

    return (
        wP * normalize(chi_P(x, y))
        + wR1 * normalize(chi_R1(x, y))
        + wR2 * normalize(chi_R2(x, y))
        + wR3 * normalize(chi_R3(x, y)) #Añado la contribución del tercer obstáculo al campo vectorial compuesto
    )

#Condiciones iniciales desde las que parte el robot.
initial_conditions = [
   [0.4, -0.2],
    [-0.3, -0.1],
    [-0.9, 1.0],
    [-0.3, -1.1],
    [0,0],
    [0.5,1.0],
    [0,1.25],
    [2.1,0]
]

#Intervalo de integración de la ecuación diferencial.
t_span = (0, 40)
t_eval = np.linspace(t_span[0], t_span[1], 3000)

solutions = []

#Integración de la ecuacion diferencial.
for z0 in initial_conditions:
    sol = solve_ivp(
        chi_composite,
        t_span,
        z0,
        t_eval=t_eval,
        rtol=1e-8,
        atol=1e-10
    )
    solutions.append(sol)

# Se crea una malla fina de puntos en el intervalo [-3, 3] para x e y.    
x = np.linspace(-3, 3, 400)
y = np.linspace(-3, 3, 400)
X, Y = np.meshgrid(x, y)

# Se evalúan las funciones escalares phi y varphi_i sobre toda la malla.
Phi = phi(X, Y)
Varphi1 = varphi1(X, Y)
Varphi2 = varphi2(X, Y)
Varphi3 = varphi3(X, Y)

# Se crea una segunda malla más gruesa para dibujar el campo vectorial y que no se solapen las flechaas entre sí.
xq = np.linspace(-3, 3, 35)
yq = np.linspace(-3, 3, 35)
Xq, Yq = np.meshgrid(xq, yq)

# Se inicializan las componentes del campo vectorial.
U = np.zeros_like(Xq)
V = np.zeros_like(Yq)

#Se evalua ne cada punto de la malla gruesa el campo vectorial compuesto, lo que devuelve la velocidad en cada punto.
for i in range(Xq.shape[0]):
    for j in range(Xq.shape[1]):
        dx, dy = chi_composite(0, [Xq[i, j], Yq[i, j]])
        U[i, j] = dx
        V[i, j] = dy

#Se calcula la norma del vector velocidad en cada punto.
N = np.sqrt(U**2 + V**2)

#Se normalizan las componentes U y V para que todas las flechas tengan longitud parecida.
U_plot = np.divide(U, N, out=np.zeros_like(U), where=N > 1e-8)
V_plot = np.divide(V, N, out=np.zeros_like(V), where=N > 1e-8)

# Se define una longitud fija para las flechas del campo vectorial.
arrow_length = 0.135
U_plot *= arrow_length
V_plot *= arrow_length

# Se dibuja el campo vectorial en cada punto (Xq,Yq).
plt.figure(figsize=(8, 8))
plt.quiver(
    Xq, Yq, U_plot, V_plot,
    angles='xy',
    scale_units='xy',
    scale=1,
    width=0.002,
    headwidth=3.5,
    headlength=4.5,
    headaxislength=4,
    pivot='middle',
    color='blue',
    alpha=0.75
)

#Dibujo del camino deseado
plt.contour(X, Y, Phi, levels=[0], colors='red', linewidths=2)

#Dibujo de las zonas reactivas de los obstáculos.
plt.contour(X, Y, Varphi1, levels=[0], colors='green', linewidths=2)
plt.contour(X, Y, Varphi2, levels=[0], colors='green', linewidths=2)
plt.contour(X, Y, Varphi3, levels=[0], colors='green', linewidths=2)

#Dibujo de las zonas repulsivas de los obstáculos.
plt.contour(X, Y, Varphi1, levels=[c], colors='black', linestyles='--', linewidths=2)
plt.contour(X, Y, Varphi2, levels=[c2], colors='black', linestyles='--', linewidths=2)
plt.contour(X, Y, Varphi3, levels=[c], colors='black', linestyles='--', linewidths=2)

#Se dibujan las trayectorias obtenidas previamente con solve_ivp.
for sol in solutions:
    plt.plot(sol.y[0], sol.y[1], linewidth=2)

#Se dibujan con puntos las condiciones iniciales.
for z0 in initial_conditions:
    plt.plot(z0[0], z0[1], 'ko', markersize=4)

ax = plt.gca()
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)

plt.grid(True)
plt.xlabel('$x (m)$', fontsize=16)
plt.ylabel('$y (m)$', fontsize=16)
plt.title('Trayectorias del robot para distintas condiciones iniciales', fontsize=20)
leyenda = [
    Line2D([0], [0], color="red", lw=2, label="Trayectoria"),
    Line2D([0], [0], color="green", lw=2, label="Zonas reactivas"),
    Line2D([0], [0], color="black", lw=2, linestyle="--", label="Zonas repulsivas"),
]

plt.legend(handles=leyenda,fontsize=12)
plt.savefig("simulacion.pdf", bbox_inches="tight")
plt.show()

